# 10. Acoustic features

## Objective

Spoken grammar proficiency can be accompanied by differences in pacing, pausing, voicing, pitch variation, and vocal energy. This notebook extracts only the acoustic features implemented in the existing code: duration; speech and silence ratios; RMS energy; zero-crossing rate; pitch statistics; voiced ratio; and pause count, mean, maximum, and total duration.

Speech duration is computed from detected non-silent intervals, with `silence_ratio = 1 - speech_ratio`. The following cells validate the extractor on one file and build the cached acoustic feature table without changing that calculation.

Audio
 │
 ├── Speech duration
 ├── Silence / pauses
 ├── Speech rate
 ├── Pitch
 ├── Energy
 └── MFCCs

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa

PROJECT_ROOT = r"C:\Users\rudra\Desktop\SHL-Grammar-Scoring"

train = pd.read_csv(
    PROJECT_ROOT + r"\train.csv"
)

print("Samples:", len(train))

In [ ]:
def extract_acoustic_features(audio_path):

    y, sr = librosa.load(audio_path, sr=16000)

    duration = len(y) / sr

    rms = librosa.feature.rms(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]

    intervals = librosa.effects.split(
        y,
        top_db=30,
        frame_length=2048,
        hop_length=512
    )

    speech_duration = np.sum(
        intervals[:, 1] - intervals[:, 0]
    ) / sr

    speech_ratio = speech_duration / duration
    silence_ratio = 1 - speech_ratio

    pause_durations = []

    for i in range(1, len(intervals)):
        pause = (
            intervals[i, 0] - intervals[i - 1, 1]
        ) / sr

        if pause > 0.2:
            pause_durations.append(pause)

    f0, voiced_flag, voiced_probs = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
        sr=sr
    )

    valid_f0 = f0[~np.isnan(f0)]

    return {
        "duration": duration,

        "speech_ratio": speech_ratio,
        "silence_ratio": silence_ratio,

        "rms_mean": np.mean(rms),
        "rms_std": np.std(rms),

        "zcr_mean": np.mean(zcr),
        "zcr_std": np.std(zcr),

        "pitch_mean": np.mean(valid_f0) if len(valid_f0) else 0,
        "pitch_std": np.std(valid_f0) if len(valid_f0) else 0,
        "pitch_range": (
            np.max(valid_f0) - np.min(valid_f0)
            if len(valid_f0) else 0
        ),

        "voiced_ratio": np.mean(voiced_flag),

        "pause_count": len(pause_durations),

        "pause_mean": (
            np.mean(pause_durations)
            if pause_durations else 0
        ),

        "pause_max": (
            np.max(pause_durations)
            if pause_durations else 0
        ),

        "pause_total": np.sum(pause_durations)
    }

In [ ]:
test_file = os.path.join(
    PROJECT_ROOT,
    "train",
    train.iloc[0]["filename"]
)

test_features = extract_acoustic_features(test_file)

print(test_features)

In [ ]:
acoustic_rows = []

for i, row in train.iterrows():

    filename = row["filename"]

    audio_path = os.path.join(
        PROJECT_ROOT,
        "train",
        filename
    )

    try:
        feats = extract_acoustic_features(audio_path)
        feats["filename"] = filename
        feats["label"] = row["label"]

        acoustic_rows.append(feats)

        if (i + 1) % 25 == 0:
            print(f"Processed {i + 1}/{len(train)}")

    except Exception as e:
        print(f"ERROR: {filename} -> {e}")

acoustic_features = pd.DataFrame(acoustic_rows)

print("\nShape:", acoustic_features.shape)
display(acoustic_features.head())

In [ ]:
acoustic_path = PROJECT_ROOT + r"\outputs\acoustic_features.csv"

acoustic_features.to_csv(acoustic_path, index=False)

print("Saved:", acoustic_path)
print("Shape:", acoustic_features.shape)

In [ ]:
print(
    acoustic_features[
        ["speech_ratio", "silence_ratio",
         "pause_count", "pause_mean",
         "pause_max", "pause_total"]
    ].describe()
)

In [ ]:
corr = acoustic_features.drop(columns=["filename"]).corr()["label"].sort_values(
    ascending=False
)

print(corr)

ACCOUSTIC ONLY

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import numpy as np

acoustic_cols = [
    "duration",
    "speech_ratio",
    "rms_mean",
    "rms_std",
    "zcr_mean",
    "zcr_std",
    "pitch_mean",
    "pitch_std",
    "pitch_range",
    "voiced_ratio",
    "pause_count",
    "pause_mean",
    "pause_max",
    "pause_total"
]

X_acoustic = acoustic_features[acoustic_cols]
y = acoustic_features["label"]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

acoustic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=10.0))
])

pred_acoustic = cross_val_predict(
    acoustic_model,
    X_acoustic,
    y,
    cv=kf
)

rmse_acoustic = np.sqrt(mean_squared_error(y, pred_acoustic))
pearson_acoustic = pearsonr(y, pred_acoustic)[0]

print("Acoustic Ridge")
print("RMSE:", rmse_acoustic)
print("Pearson:", pearson_acoustic)

E6 — Linguistic + Acoustic
This is the important one.
We already have:
- linguistic_features
- acoustic_features
We need to merge them by filename.

In [ ]:
import pandas as pd

ling_path = PROJECT_ROOT + r"\outputs\linguistic_features.csv"
acoustic_path = PROJECT_ROOT + r"\outputs\acoustic_features.csv"

linguistic_features = pd.read_csv(ling_path)
acoustic_features = pd.read_csv(acoustic_path)

print("Linguistic:", linguistic_features.shape)
print("Acoustic:", acoustic_features.shape)

In [ ]:
ling = linguistic_features.copy()
acoustic = acoustic_features.copy()

combined = ling.merge(
    acoustic,
    on=["filename", "label"],
    how="inner"
)

print("Combined shape:", combined.shape)

In [ ]:
ling_cols = [
    "word_count",
    "unique_word_count",
    "sentence_count",
    "avg_sentence_length",
    "sentence_length_std",
    "type_token_ratio",
    "char_count",
    "avg_word_length",
    "repetition_ratio",
    "filler_count",
    "question_count",
    "exclamation_count"
]

acoustic_cols = [
    "duration",
    "speech_ratio",
    "rms_mean",
    "rms_std",
    "zcr_mean",
    "zcr_std",
    "pitch_mean",
    "pitch_std",
    "pitch_range",
    "voiced_ratio",
    "pause_count",
    "pause_mean",
    "pause_max",
    "pause_total"
]

feature_cols = ling_cols + acoustic_cols

X_combined = combined[feature_cols]
y_combined = combined["label"]

combined_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=10.0))
])

pred_combined = cross_val_predict(
    combined_model,
    X_combined,
    y_combined,
    cv=kf
)

rmse_combined = np.sqrt(
    mean_squared_error(y_combined, pred_combined)
)

pearson_combined = pearsonr(
    y_combined,
    pred_combined
)[0]

print("Linguistic + Acoustic Ridge")
print("RMSE:", rmse_combined)
print("Pearson:", pearson_combined)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import numpy as np

alphas = [0.1, 1, 3, 10, 30, 100]

results = []

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for alpha in alphas:

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])

    pred = cross_val_predict(
        model,
        X_combined,
        y_combined,
        cv=kf
    )

    rmse = np.sqrt(
        mean_squared_error(y_combined, pred)
    )

    pearson = pearsonr(
        y_combined,
        pred
    )[0]

    results.append({
        "alpha": alpha,
        "RMSE": rmse,
        "Pearson": pearson
    })

results_df = pd.DataFrame(results)

print(results_df.sort_values("RMSE"))

CHECK FOR DATA LEAKAGE


In [ ]:
import pandas as pd

train = pd.read_csv(PROJECT_ROOT + r"\train.csv")
transcripts = pd.read_csv(
    PROJECT_ROOT + r"\outputs\train_transcripts.csv"
)

data = train.merge(
    transcripts[["filename", "transcript"]],
    on="filename",
    how="left"
)

print("Duplicate filenames:", data["filename"].duplicated().sum())
print("Duplicate transcripts:", data["transcript"].duplicated().sum())

In [ ]:
dup_transcripts = (
    data.groupby("transcript")["label"]
    .nunique()
    .sort_values(ascending=False)
)

print("Transcripts associated with >1 label:")
print(dup_transcripts[dup_transcripts > 1])

In [ ]:
print(train["filename"].head(30).tolist())

In [ ]:
print(train["filename"].tail(30).tolist())

In [ ]:
import re

for pattern in [
    r"speaker[_-]?(\d+)",
    r"spk[_-]?(\d+)",
    r"subject[_-]?(\d+)",
    r"person[_-]?(\d+)",
    r"(\d+)"
]:
    matches = train["filename"].str.extract(pattern, expand=False)
    print(pattern, "->", matches.notna().sum(), "matches")

In [ ]:
import spacy

print(spacy.__version__)

In [ ]:
print(spacy.util.get_installed_models())

## Acoustic-model result

The recorded acoustic-only Ridge sweep reaches its best listed result at `alpha=30`: RMSE `0.9506462573030953` and Pearson `0.6408753565367712`. This is an evaluation result from the saved experiment table, not a new training run in this notebook. Acoustic information therefore contributes a meaningful complementary signal and is carried into the final feature set.